In [1]:
# 6-18-2026

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import regionmask
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler

In [3]:
zarr_path = "seasfire_pyromes_ecoregions.zarr"

In [6]:
ds = xr.open_zarr(zarr_path, consolidated=True)
df = pd.read_csv("ecoregions_stats.csv")

In [7]:
ds

<xarray.Dataset> Size: 76GB
Dimensions:                         (latitude: 720, longitude: 1440, time: 506)
Coordinates:
  * latitude                        (latitude) float64 6kB 89.88 ... -89.88
  * longitude                       (longitude) float64 12kB -179.9 ... 179.9
  * time                            (time) datetime64[ns] 4kB 2011-01-01 ... ...
Data variables: (12/43)
    area                            (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    biomes                          (latitude, longitude) float32 4MB dask.array<chunksize=(45, 45), meta=np.ndarray>
    cams_co2fire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    cams_frpfire                    (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_max                (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    drought_code_mean               (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ...                              ...
    t2m_max                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_mean                        (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    t2m_min                         (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    tp                              (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    vpd                             (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
    ws10                            (time, latitude, longitude) float32 2GB dask.array<chunksize=(506, 45, 45), meta=np.ndarray>
Attributes:
    crs:          EPSG:4326
    description:  The SeasFire Cube is a scientific datacube for seasonal fir...
    title:        SeasFire Cube: A Global Dataset for Seasonal Fire Modeling ...

In [8]:
df

,ecoregion_id,ndvi_mean,ndvi_std,tp_mean,tp_std,t2m_mean_mean,t2m_mean_std,vpd_mean,vpd_std,fwi_mean_mean,fwi_mean_std
0,10101.0,0.861981,0.018632,87.606900,46.894028,300.05972,0.495307,7.156866,1.218896,0.523723,1.157128
1,10102.0,0.811376,0.056275,41.696075,44.840942,299.99167,0.916401,8.344715,1.445781,5.379829,7.148313
2,10103.0,0.831247,0.016678,73.235190,37.511982,300.35510,0.404653,7.995025,0.804152,0.309281,0.589486
3,10104.0,0.818266,0.037402,75.606750,50.415367,297.25217,1.323315,5.550790,1.390918,1.663618,4.157984
4,10105.0,0.804765,0.037594,133.072220,83.581340,293.26993,3.177103,4.774403,1.419347,0.165741,0.525632
...,...,...,...,...,...,...,...,...,...,...,...
786,81329.0,0.096504,0.013685,0.478033,2.007010,300.93298,6.504858,32.736767,12.275587,76.862015,16.856901
787,81330.0,0.079987,0.063435,2.011220,5.163428,284.36990,13.093088,13.684868,10.943560,43.847490,23.136137
788,81331.0,0.089542,0.012297,0.756772,4.336491,295.88806,6.333597,24.953905,9.702799,63.496370,14.658874
789,81332.0,0.100678,0.014361,0.659456,2.479339,297.66815,7.632448,28.720503,12.799202,71.168530,17.144644


In [16]:
feature_cols = [
    "ndvi_mean",
    "ndvi_std",
    "tp_mean",
    "tp_std",
    "t2m_mean_mean",
    "t2m_mean_std",
    "vpd_mean",
    "vpd_std",
    "fwi_mean_mean",
    "fwi_mean_std"
]

df_scaled = df.copy()

In [17]:
df_scaled["ecoregion_id"] = df_scaled["ecoregion_id"].astype(int) # original had .0 at end 

In [18]:
scaler = StandardScaler()

df_scaled[feature_cols] = scaler.fit_transform(
    df_scaled[feature_cols]
)

df_scaled.head()

,ecoregion_id,ndvi_mean,ndvi_std,tp_mean,tp_std,t2m_mean_mean,t2m_mean_std,vpd_mean,vpd_std,fwi_mean_mean,fwi_mean_std
0,10101,1.583927,-1.436430,2.349207,0.818652,0.974691,-1.223407,-0.313550,-0.973492,-0.894021,-1.364158
1,10102,1.364330,-0.937810,0.478327,0.714218,0.968130,-1.122366,-0.098350,-0.901506,-0.569004,-0.546033
2,10103,1.450560,-1.462310,1.763555,0.341418,1.003170,-1.245159,-0.161702,-1.105081,-0.908373,-1.441672
3,10104,1.394231,-1.187800,1.860197,0.997771,0.704001,-1.024727,-0.604520,-0.918913,-0.817728,-0.954377
4,10105,1.335644,-1.185254,4.201932,2.684818,0.320053,-0.579913,-0.745177,-0.909893,-0.917980,-1.450391


In [19]:
df_scaled[feature_cols].mean()

ndvi_mean       -1.361193e-16
ndvi_std         5.444772e-17
tp_mean         -4.491421e-17
tp_std           3.593136e-17
t2m_mean_mean   -3.754828e-15
t2m_mean_std     1.077941e-16
vpd_mean         1.796568e-17
vpd_std         -1.976225e-16
fwi_mean_mean   -7.259696e-17
fwi_mean_std    -2.132536e-16
dtype: float64

In [ ]:
df_scaled[feature_cols].std() # perfect

ndvi_mean        1.000639
ndvi_std         1.000639
tp_mean          1.000633
tp_std           1.000633
t2m_mean_mean    1.000633
t2m_mean_std     1.000633
vpd_mean         1.000633
vpd_std          1.000633
fwi_mean_mean    1.000639
fwi_mean_std     1.000639
dtype: float64